# Optimal Locations for EV Charging Stations in Amsterdam

This notebook walks step by step through the spatial analysis (using
GeoPandas) to determine the best locations for new electric vehicle
charging stations in Amsterdam, based on **real open data**.

**Method:**
1. Load neighborhood boundaries, existing charging stations, and population density
2. Build a 250 m coverage buffer around existing charging stations
3. Identify the areas outside this coverage buffer
4. Identify neighborhoods with high population density but low charging
   coverage
5. Generate candidate points within these neighborhoods, score them, and
   select the best 10
6. Display the results on an interactive Folium map

> **Data sources used:**
> - **Neighborhood boundaries**: City of Amsterdam, neighborhood classification with CBS neighborhood codes (517 neighborhoods)
> - **Population**: CBS (Statistics Netherlands), "Key figures for districts and neighborhoods" (table 86165NED), number of residents per neighborhood
> - **Existing charging stations**: Open Charge Map (OCM), 673 charging locations in Amsterdam
>
> The raw source files are in `data/raw/`; `src/process_real_data.py`
> converts them into the usable files in `data/real/` that this notebook
> uses. See the README for details.

In [ ]:
import sys
sys.path.append('../src')

import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

from analyse import (
    laad_data as load_data,
    buffer_bestaande_laadpalen as buffer_existing_chargers,
    onbedekt_gebied as uncovered_area,
    bepaal_prioriteitsbuurten as determine_priority_neighborhoods,
    genereer_kandidaatpunten as generate_candidate_points,
    score_kandidaten as score_candidates,
    verwijder_te_dichtbij as remove_too_close,
    top_n_locaties as top_n_locations
)

DATA_DIR = '../data/echt'
plt.rcParams['figure.figsize'] = (9, 9)

# English column names used throughout this notebook, mapped to the
# original Dutch column names produced by the source data / scripts
COLUMN_MAP = {
    'buurt_naam': 'neighborhood_name',
    'stadsdeel': 'district',
    'bevolking': 'population',
    'oppervlakte_km2': 'area_km2',
    'bev_dichtheid': 'pop_density',
    'pct_onbedekt': 'pct_uncovered',
    'afstand_tot_dichtstbijzijnde_laadpaal_m': 'dist_to_nearest_charger_m',
    'rangorde': 'rank',
}

def to_en(gdf):
    """Rename any Dutch columns present in gdf to their English equivalents."""
    return gdf.rename(columns={k: v for k, v in COLUMN_MAP.items() if k in gdf.columns})

## 1. Load data

In [ ]:
neighborhoods, chargers = load_data(
    f'{DATA_DIR}/buurten_amsterdam.geojson',
    f'{DATA_DIR}/ev_laadpalen_bestaand.geojson'
)
neighborhoods = to_en(neighborhoods)
chargers = to_en(chargers)

print(f'{len(neighborhoods)} neighborhoods, {len(chargers)} existing charging stations loaded')
neighborhoods.head()

In [ ]:
fig, ax = plt.subplots()
neighborhoods.plot(column='pop_density', cmap='YlOrRd', legend=True, ax=ax,
                    edgecolor='grey', linewidth=0.3)
chargers.plot(ax=ax, color='blue', markersize=3, alpha=0.6)
ax.set_title('Population density (neighborhoods) and existing charging stations')
ax.set_axis_off()
plt.show()

## 2. 250 m buffer and areas outside coverage

In [ ]:
coverage = buffer_existing_chargers(chargers, buffer_m=250)
gap = uncovered_area(neighborhoods, coverage)
gap = to_en(gap)

fig, ax = plt.subplots()
gap.plot(column='pct_uncovered', cmap='Reds', legend=True, ax=ax,
         edgecolor='grey', linewidth=0.3)
coverage.boundary.plot(ax=ax, color='blue', linewidth=0.5)
ax.set_title('Percentage of neighborhood area outside coverage')
ax.set_axis_off()
plt.show()

gap[['neighborhood_name','district','pop_density','pct_uncovered']].sort_values('pct_uncovered', ascending=False).head(10)

## 3. Priority neighborhoods

Neighborhoods with a population density above the median **and** where
at least 35% of the area falls outside the 250 m coverage buffer =
priority areas for new charging stations.

In [ ]:
priority = determine_priority_neighborhoods(gap, dichtheid_percentiel=0.4, min_pct_onbedekt=35.0)
priority = to_en(priority)
print(f'{len(priority)} priority neighborhoods found')
priority[['neighborhood_name','district','pop_density','pct_uncovered']]

## 4. Generate and score candidate points

Score = 0.45 × density + 0.35 × distance to the nearest charging station + 0.20 × share uncovered (all normalized to 0-1, on a scale of 100).

In [ ]:
candidates = generate_candidate_points(priority, punten_per_buurt=4)
scored = score_candidates(candidates, chargers)
spread_out = remove_too_close(scored, min_afstand_m=300)
spread_out = to_en(spread_out)

print(f'{len(candidates)} candidates generated -> {len(spread_out)} candidates (after 300 m mutual-distance filter)')
spread_out.head(10)[['neighborhood_name','district','pop_density','dist_to_nearest_charger_m','score']]

## 5. Result: top 10 best new charging station locations

In [ ]:
top10, layers = top_n_locations(
    f'{DATA_DIR}/buurten_amsterdam.geojson',
    f'{DATA_DIR}/ev_laadpalen_bestaand.geojson',
    n=10
)
top10 = to_en(top10)
top10[['rank','neighborhood_name','district','pop_density','dist_to_nearest_charger_m','pct_uncovered','score']]

In [ ]:
fig, ax = plt.subplots()
neighborhoods.plot(ax=ax, color='#f0f0f0', edgecolor='grey', linewidth=0.3)
coverage.plot(ax=ax, color='#2b8cbe', alpha=0.25)
chargers.plot(ax=ax, color='#2b8cbe', markersize=3, alpha=0.5, label='Existing charging station')
top10.plot(ax=ax, color='green', markersize=60, marker='*', label='Recommended new location (top 10)')
ax.set_title('Recommended top 10 new EV charging station locations')
ax.set_axis_off()
plt.legend()
plt.show()

## 6. Interactive Folium map

An interactive map with all layers (population density, coverage area, existing charging stations, recommended top 10) is saved as `outputs/interactieve_kaart.html` and is also shown directly below in the notebook.

In [ ]:
import sys
sys.path.append('../src')
from bouw_kaart import bouw_kaart as build_map

map_path = build_map()

from IPython.display import IFrame
IFrame(src='../outputs/interactieve_kaart.html', width='100%', height=600)

## Exporting results

The top 10 locations are saved in `outputs/top10_kandidaten.csv`, the
map in `outputs/interactieve_kaart.html`. These files can also be
loaded directly into QGIS (CSV -> "Add Delimited Text Layer",
choose the lat/lon columns as X/Y).